In [1]:
# Ejercicio 1

## Importaciones

In [26]:
# Importamos las bibliotecas que usaremos
from sklearn.feature_extraction.text import CountVectorizer  # Para crear bag-of-words
from sklearn.decomposition import LatentDirichletAllocation  # Para el modelo LDA
import numpy as np  # Para operaciones numéricas
import pandas as pd  # Para manipular datos tabulares
import json, re
from nltk.corpus import stopwords
import nltk
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')  # Suprimimos warnings para mayor claridad

## Carga de datos

In [22]:
# Lista de textos a procesar
with open('/mnt/c/Users/carme/Desktop/PLN/T3_Listados/Tema3Hoja2/Noticias.json', encoding="utf8") as json_file:
    datos = json.load(json_file)

tuplas = list(zip([noticia.get("Title") for noticia in datos],[noticia.get("TextContent") for noticia in datos]))
df = pd.DataFrame(tuplas, columns =['Titular', 'Noticia'])


## Procesamiento inicial

In [27]:
# Lista de textos a procesar
documents = df['Noticia'].tolist()

nltk.download('stopwords')

# Configurar y crear el vectorizador
tf_vectorizer = CountVectorizer(
    stop_words=stopwords.words('spanish'),  # No eliminamos stopwords por ahora []
    min_df=1,      # Incluir palabras que aparecen al menos 1 vez
    max_df=1.0,    # Sin límite superior de frecuencia
    lowercase=True, # Convertir todo a minúsculas
    max_features=50000,  # Máximo número de palabras a considerar
    token_pattern='[a-zA-Z0-9]{3,}',  # Palabras de 3+ caracteres
    analyzer = 'word'
)

# Crear la matriz de documentos-términos
bag_of_words = tf_vectorizer.fit_transform(documents)

# Obtener el vocabulario
dictionary = tf_vectorizer.get_feature_names_out()
vocabulary = tf_vectorizer.vocabulary_

print("Estadísticas del preprocesamiento:")
print(f"- Tamaño del vocabulario: {len(dictionary)} palabras únicas")
print(f"- Dimensiones de la matriz: {bag_of_words.shape}")

# Mostrar las palabras más frecuentes
word_freq = bag_of_words.sum(axis=0).A1
top_words_idx = word_freq.argsort()[-10:][::-1]
print("\nPalabras más frecuentes:")
for idx in top_words_idx:
    print(f"- {dictionary[idx]}: {word_freq[idx]} apariciones")

Estadísticas del preprocesamiento:
- Tamaño del vocabulario: 359 palabras únicas
- Dimensiones de la matriz: (12, 359)

Palabras más frecuentes:
- ses: 4 apariciones
- horas: 4 apariciones
- estatal: 3 apariciones
- verdad: 3 apariciones
- meteorolog: 3 apariciones
- provincia: 3 apariciones
- ayuda: 3 apariciones
- laga: 3 apariciones
- aemet: 3 apariciones
- litros: 3 apariciones


[nltk_data] Downloading package stopwords to /home/vc/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Entrenamiento LDA

El algoritmo LDA tiene varios hiperparámetros importantes:

* n_topics: Número de tópicos a encontrar
    - Debe elegirse según el conocimiento del dominio
    - Se puede optimizar usando métricas como coherencia o perplejidad

* alpha: Prior de la distribución documentos-tópicos
    - alpha < 1: documentos se concentran en pocos tópicos
    - alpha > 1: documentos mezclan varios tópicos
    - alpha = 1: distribución uniforme

* beta: Prior de la distribución tópicos-palabras
    - beta < 1: tópicos más específicos (pocas palabras)
    - beta > 1: tópicos más generales (muchas palabras)
    - beta = 1: distribución uniforme





In [30]:
# Parámetros del modelo
n_topics = 4    # Número moderado de tópicos para empezar
alpha = 1.0     # Documentos algo especializados
beta = 0.1     # Tópicos bastante específicos

# Crear y entrenar el modelo
print("Configuración del modelo LDA:")
print(f"- Número de tópicos: {n_topics}")
print(f"- Alpha: {alpha}")
print(f"- Beta: {beta}")
print("\nIniciando entrenamiento...\n")

lda = LatentDirichletAllocation(
    n_components=n_topics,      # Número de tópicos
    doc_topic_prior=alpha,      # Prior documentos-tópicos
    topic_word_prior=beta,      # Prior tópicos-palabras
    max_iter=25,               # Máximo de iteraciones
    learning_method='online',   # Método de aprendizaje
    evaluate_every=1,          # Evaluar en cada iteración
    n_jobs=-1,                # Usar todos los cores
    random_state=0,           # Semilla para reproducibilidad
    verbose=1                 # Mostrar progreso
)

# Entrenar el modelo
lda.fit(bag_of_words)

Configuración del modelo LDA:
- Número de tópicos: 4
- Alpha: 1.0
- Beta: 0.1

Iniciando entrenamiento...

iteration: 1 of max_iter: 25, perplexity: 20387.7060
iteration: 2 of max_iter: 25, perplexity: 16036.8688
iteration: 3 of max_iter: 25, perplexity: 12300.9210
iteration: 4 of max_iter: 25, perplexity: 9420.4856
iteration: 5 of max_iter: 25, perplexity: 7304.1397
iteration: 6 of max_iter: 25, perplexity: 5786.3557
iteration: 7 of max_iter: 25, perplexity: 4692.0171
iteration: 8 of max_iter: 25, perplexity: 3890.3933
iteration: 9 of max_iter: 25, perplexity: 3292.6657
iteration: 10 of max_iter: 25, perplexity: 2839.3214
iteration: 11 of max_iter: 25, perplexity: 2490.1006
iteration: 12 of max_iter: 25, perplexity: 2217.3174
iteration: 13 of max_iter: 25, perplexity: 2001.5875
iteration: 14 of max_iter: 25, perplexity: 1829.0975
iteration: 15 of max_iter: 25, perplexity: 1689.8372
iteration: 16 of max_iter: 25, perplexity: 1576.4380
iteration: 17 of max_iter: 25, perplexity: 1483.396

,n_components,4
,doc_topic_prior,1.0
,topic_word_prior,0.1
,learning_method,'online'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,25
,batch_size,128
,evaluate_every,1
,total_samples,1000000.0
,perp_tol,0.1


## Análisis de Resultados

Se analizan los resultados de tres formas diferentes:

* Palabras más relevantes por tópico
* Documentos más representativos de cada tópico
* Distribución de tópicos en documentos específicos



In [31]:
# Configuración de visualización
no_top_words = 10      # Número de palabras top por tópico
no_top_documents = 5   # Número de documentos top por tópico

# Obtener las distribuciones
doc_topics = lda.transform(bag_of_words)  # Distribución de tópicos por documento
topics = lda.components_                  # Distribución de palabras por tópico


# 1. Detallar tópicos encontrados
print("TÓPICOS DESCUBIERTOS")
print("Cada tópico se representa por sus palabras más probables\n")

for topic_idx, topic in enumerate(topics):
    print(f" Tópico {topic_idx + 1}:")
    # Obtener índices de las palabras más probables
    top_words_idx = topic.argsort()[:-no_top_words-1:-1]
    top_words = [dictionary[i] for i in top_words_idx]
    top_probs = [topic[i] for i in top_words_idx]

    # Mostrar palabras y sus probabilidades
   
    for word, prob in zip(top_words, top_probs):
        print(f"   {word}: {prob:.4f}")
    print()



TÓPICOS DESCUBIERTOS
Cada tópico se representa por sus palabras más probables

 Tópico 1:
   litros: 2.0814
   beckett: 2.0398
   vez: 2.0385
   fracasa: 2.0365
   metro: 1.1044
   menos: 1.0995
   cuadrado: 1.0988
   igual: 1.0973
   godot: 1.0955
   durar: 1.0954

 Tópico 2:
   horas: 2.2304
   laga: 2.1407
   rojo: 2.1211
   afectados: 2.0402
   seguridad: 2.0161
   aemet: 1.1696
   meteorolog: 1.1636
   provincia: 1.1583
   estatal: 1.1558
   agencia: 1.1324

 Tópico 3:
   ses: 3.1134
   ucrania: 2.2145
   ayuda: 2.1720
   reino: 2.0503
   kiev: 2.0476
   gobierno: 2.0449
   canad: 2.0423
   unido: 2.0404
   militar: 1.1585
   mosc: 1.1451

 Tópico 4:
   verdad: 2.0461
   ser: 2.0461
   cula: 2.0435
   industria: 2.0355
   podr: 2.0319
   pel: 2.0286
   aunque: 2.0228
   necesario: 1.1072
   distingue: 1.0941
   alguna: 1.0940



In [32]:
# 2. Documentos más representativos por tópico
print("\n DOCUMENTOS MÁS REPRESENTATIVOS POR TÓPICO")
print("Se muestran los documentos que más peso tienen en cada tópico\n")

for topic_idx in range(n_topics):
    print(f" Tópico {topic_idx + 1}:")
    # Obtener los documentos más representativos
    top_doc_indices = np.argsort(doc_topics[:,topic_idx])[::-1][:no_top_documents]

    for doc_idx in top_doc_indices:
        title = df.iloc[doc_idx]['Titular']
        weight = doc_topics[doc_idx, topic_idx]
        print(f"   '{title}'")
        print(f"      Peso: {weight:.4f}")
    print()


 DOCUMENTOS MÁS REPRESENTATIVOS POR TÓPICO
Se muestran los documentos que más peso tienen en cada tópico

 Tópico 1:
   'Emilia Pérez, Karla Sofía Gascón, Demi Moore y la decencia, a la cabeza de los perdedores de la noche'
      Peso: 0.9233
   'La Aemet retira también el aviso rojo por fuertes lluvias en Castellón, que se mantiene en naranja.'
      Peso: 0.9016
   'Trump congela toda la ayuda militar a Ucrania para castigar y doblegar a Zelenski'
      Peso: 0.8892
   'Valladolid ya no permite más terrazas cerradas y niega las últimas once licencias.'
      Peso: 0.8499
   'Objetivo: acabar con la pesca fantasma y las redes abandonadas que matan y mutilan tortugas en el Mediterráneo'
      Peso: 0.0500

 Tópico 2:
   'Suspendido el partido Villarreal-Espanyol por la emergencia meteorológica.'
      Peso: 0.9453
   'España propone financiar la defensa de los países de la UE con fondos europeos'
      Peso: 0.8927
   'Objetivo: acabar con la pesca fantasma y las redes abandonadas que

In [33]:
# Para mostrar matriz documento-tópico
from IPython.display import display, HTML
import pandas as pd
#pd.set_option('display.max_columns', None)

topicnames = ["topic"+ str(x) for x in range(0, lda.n_components)]
norm_doc_topics = []
for i in doc_topics:
  norm_doc_topics.append([ "{0:.3f}".format(weight) for weight in i])

df = pd.DataFrame(norm_doc_topics,
                  columns=topicnames,
                  index=df['Titular'].tolist())

df

,topic0,topic1,topic2,topic3
Suspendido el partido Villarreal-Espanyol por la emergencia meteorológica.,0.019,0.945,0.017,0.018
Reino Unido y otros países aliados de Ucrania se comprometen a rearmar a Zelenski: 'Botas en el terreno y aviones en los cielos',0.017,0.017,0.949,0.017
Los premios Oscar dan la gloria al cine indie y castigan una vez más a Netflix,0.015,0.014,0.014,0.957
"Emilia Pérez, Karla Sofía Gascón, Demi Moore y la decencia, a la cabeza de los perdedores de la noche",0.923,0.025,0.026,0.026
"La Aemet retira también el aviso rojo por fuertes lluvias en Castellón, que se mantiene en naranja.",0.902,0.042,0.026,0.030
España propone financiar la defensa de los países de la UE con fondos europeos,0.033,0.893,0.039,0.035
El tequila lubricó la gran noche en la que Hollywood no quiso hablar de Donald Trump,0.029,0.031,0.031,0.909
Trump congela toda la ayuda militar a Ucrania para castigar y doblegar a Zelenski,0.889,0.032,0.046,0.032
"La Comunidad Valenciana y otras cinco regiones, bajo aviso por lluvia",0.026,0.038,0.021,0.915
Objetivo: acabar con la pesca fantasma y las redes abandonadas que matan y mutilan tortugas en el Mediterráneo,0.050,0.053,0.050,0.847


In [17]:
# Matriz tópico-palabra

# Topic-Keyword Matrix
df_topic_keywords = pd.DataFrame(lda.components_ / lda.components_.sum(axis=1)[:, np.newaxis])

# Assign Column and Index
df_topic_keywords.columns = dictionary
df_topic_keywords.index = topicnames

# View
df_topic_keywords.head()

,000,100,106,143,180,2017,2024,500,abundancia,academia,...,vieron,viles,volod,voluntad,willem,ximo,xito,zelenski,zonas,zorrilla
topic0,0.001672,0.001621,0.001706,0.001546,0.001586,0.001594,0.001618,0.001587,0.001607,0.001593,...,0.001548,0.001647,0.001584,0.001462,0.001574,0.001734,0.001529,0.001510,0.001588,0.001589
topic1,0.001956,0.002073,0.001894,0.002028,0.002006,0.001934,0.001897,0.001977,0.001901,0.001921,...,0.001883,0.001996,0.001943,0.001990,0.001991,0.001957,0.001952,0.002014,0.001971,0.002037
topic2,0.004233,0.004233,0.000576,0.000599,0.000555,0.000569,0.000554,0.000605,0.000551,0.000517,...,0.000653,0.000591,0.000544,0.000560,0.000543,0.000596,0.004187,0.000564,0.004200,0.004142
topic3,0.000873,0.000871,0.006523,0.006451,0.006602,0.000816,0.000865,0.006549,0.000872,0.000837,...,0.006551,0.006527,0.000890,0.000887,0.000883,0.006504,0.000867,0.000899,0.000909,0.000954
topic4,0.001178,0.001081,0.001187,0.001183,0.001112,0.001154,0.001135,0.001140,0.008554,0.001168,...,0.001126,0.001050,0.008285,0.008416,0.001123,0.001249,0.001162,0.008418,0.001067,0.001192


## Evaluación del modelo

Utilizamos dos métricas principales:

1. Log Likelihood (mayor es mejor):
   * Indica cómo de bien el modelo explica los datos
   * Valores más altos indican mejor ajuste

2. Perplejidad (menor es mejor):
   * Mide qué tan "sorprendido" está el modelo por los datos
   * Valores más bajos indican mejor generalización



In [34]:
# Calcular métricas
log_likelihood = lda.score(bag_of_words)
perplexity = lda.perplexity(bag_of_words)

print("MÉTRICAS DE EVALUACIÓN")
print(f"- Log Likelihood: {log_likelihood:.2f}")
print(f"- Perplejidad: {perplexity:.2f}")

# Comparar con diferentes valores de hiperparámetros
print("\nCOMPARACIÓN DE HIPERPARÁMETROS")
print("Probando diferentes configuraciones para encontrar el mejor modelo...")

# Probar diferentes números de tópicos
n_topics_range = [2,4,6,8]
results = []

for n_top in n_topics_range:
    model = LatentDirichletAllocation(
        n_components=n_top,
        doc_topic_prior=alpha,
        topic_word_prior=beta,
        max_iter=25,
        random_state=0
    )
    model.fit(bag_of_words)

    results.append({
        'n_topics': n_top,
        'perplexity': model.perplexity(bag_of_words),
        'log_likelihood': model.score(bag_of_words)
    })

# Mostrar resultados
results_df = pd.DataFrame(results)
print("\nResultados con diferentes números de tópicos:")
print(results_df)

MÉTRICAS DE EVALUACIÓN
- Log Likelihood: -2992.85
- Perplejidad: 1124.91

COMPARACIÓN DE HIPERPARÁMETROS
Probando diferentes configuraciones para encontrar el mejor modelo...

Resultados con diferentes números de tópicos:
   n_topics   perplexity  log_likelihood
0         2  1100.268804    -2983.409973
1         4   925.491364    -2909.718365
2         6   897.154055    -2896.470954
3         8   948.214699    -2920.051486


# Ejercicio 3 Statistic Language Model
Utilizando n-gramas y el corpus Reuters de NLTK se van a predecir palabras. Se pide lo 
siguiente: 
1. Imprimir el número completo de trigramas de palabras de todas las sentencias 
del corpus.  
2. Imprimir el número de veces que la palabra “economists” sigue a “what the”.  
3. ¿Y  si  preguntamos  por  una  palabra  que  no  existe?  Por  ejemplo,  imprimir  el 
número de veces que “nonexistingword” sigue a “what the”.  
4. ¿Cómo saber cuántas oraciones comienzan por “The”? 
5. Se  pueden  transformar  las  frecuencias  en  probabilidades  para  responder  a  las 
preguntas 3 y 4 con probabilidades.  
6. Si quisiéramos generar texto se podría. ¿Cuáles serían las palabras más 
probables que siguen a “The Market”? 
 
Nota: Utilizar la clase DefaultDict de Python (from collections import defaultdict) 
 

In [2]:
import nltk
nltk.download("reuters")
nltk.download("punkt")


[nltk_data] Downloading package reuters to /home/vc/nltk_data...
[nltk_data] Downloading package punkt to /home/vc/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
from nltk.corpus import reuters
from nltk.util import ngrams
from collections import defaultdict, Counter

# Crear modelo de trigramas
trigram_model = defaultdict(list)
trigram_freq = Counter()

for sentence in reuters.sents():
    tokens = [w.lower() for w in sentence]
    for trigram in ngrams(tokens, 3):
        trigram_model[(trigram[0], trigram[1])].append(trigram[2])
        trigram_freq[trigram] += 1

# Resultados
print("Total de trigramas:", sum(trigram_freq.values()))
print('Veces que "economists" sigue a "what the":', trigram_freq[("what", "the", "economists")])
print('Veces que "nonexistingword" sigue a "what the":', trigram_freq[("what", "the", "nonexistingword")])

# Contar oraciones que empiezan por "The" en el corpus de Reuters
contador = 0

# Recorremos todas las oraciones del corpus
for oracion in reuters.sents():
    if len(oracion) > 0:  # Asegurarse de que la oración no esté vacía
        primera_palabra = oracion[0].lower()
        if primera_palabra == "the":
            contador += 1

print(f'Oraciones que empiezan por "The": {contador}')

Total de trigramas: 1611527
Veces que "economists" sigue a "what the": 2
Veces que "nonexistingword" sigue a "what the": 0
Oraciones que empiezan por "The": 8865
